In [6]:
!pip install lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=8ab6e167e01e11ea0e846b76b61863977f44544cbdd187dcb7715c5485e8e453
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [7]:
# ================================================================
# Mutahhir CYBERSECURITY PROJECT
# Explainable Machine Learning-Based Intrusion Detection System
# ================================================================
#
# Project Workflow:
#
# Enterprise Network Traffic
#          ↓
# CICIDS2017 Dataset
#          ↓
# Data Cleaning and Pre-processing
#          ↓
# Feature Selection and Normalisation
#          ↓
# Random Forest + XGBoost + SVM
#          ↓
# Intrusion Prediction
#          ↓
# Explainability using SHAP + LIME
#          ↓
# Analyst Decision Support
#
# ================================================================
#
# Install packages first:
#
# pip install pandas numpy matplotlib scikit-learn xgboost shap lime joblib
#
# ================================================================


# ================================================================
# 1. IMPORT LIBRARIES
# ================================================================

import os
import json
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score
)

from sklearn.preprocessing import label_binarize

from xgboost import XGBClassifier

import joblib


# SHAP
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP is not installed.")
    print("Install using: pip install shap")


# LIME
try:
    from lime.lime_tabular import LimeTabularExplainer
    LIME_AVAILABLE = True
except ImportError:
    LIME_AVAILABLE = False
    print("LIME is not installed.")
    print("Install using: pip install lime")


# ================================================================
# 2. PROJECT CONFIGURATION
# ================================================================

RANDOM_STATE = 42

TEST_SIZE = 0.20

TOP_K_FEATURES = 20

DATA_FILE = "Mutahhir_CICIDS2017_30000.csv"


# Create output directories

OUTPUT_FOLDER = "outputs"

FIGURE_FOLDER = os.path.join(
    OUTPUT_FOLDER,
    "figures"
)

TABLE_FOLDER = os.path.join(
    OUTPUT_FOLDER,
    "tables"
)

MODEL_FOLDER = os.path.join(
    OUTPUT_FOLDER,
    "models"
)


os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)

os.makedirs(
    FIGURE_FOLDER,
    exist_ok=True
)

os.makedirs(
    TABLE_FOLDER,
    exist_ok=True
)

os.makedirs(
    MODEL_FOLDER,
    exist_ok=True
)


# ================================================================
# 3. LOAD CICIDS2017 DATASET
# ================================================================

print("\n================================================")
print("STEP 1: DATASET COLLECTION")
print("================================================")


if not os.path.exists(DATA_FILE):

    raise FileNotFoundError(
        f"\nDataset not found: {DATA_FILE}\n"
        f"Please place {DATA_FILE} in the same folder "
        "as this Python program."
    )


df = pd.read_csv(DATA_FILE)


print("\nDataset successfully loaded.")

print(
    "\nDataset dimensions:",
    df.shape
)

print(
    "\nNumber of rows:",
    len(df)
)

print(
    "\nNumber of columns:",
    len(df.columns)
)


print("\nDataset columns:\n")

for column in df.columns:
    print(column)


print("\nFirst five observations:\n")

print(
    df.head()
)


# ================================================================
# 4. ORIGINAL CLASS DISTRIBUTION
# ================================================================

print("\n================================================")
print("TRAFFIC CLASS DISTRIBUTION")
print("================================================")


print(
    df["Label"].value_counts()
)


class_distribution = (
    df["Label"]
    .value_counts()
    .reset_index()
)

class_distribution.columns = [
    "Traffic_Class",
    "Number_of_Flows"
]


class_distribution.to_csv(
    os.path.join(
        TABLE_FOLDER,
        "class_distribution.csv"
    ),
    index=False
)


# ================================================================
# 5. DATA CLEANING
# ================================================================

print("\n================================================")
print("STEP 2: DATA CLEANING AND PRE-PROCESSING")
print("================================================")


# Remove spaces around column names

df.columns = [
    column.strip()
    for column in df.columns
]


# ------------------------------------------------
# Check missing values
# ------------------------------------------------

print(
    "\nMissing values before cleaning:"
)

print(
    df.isnull()
    .sum()
    .sort_values(
        ascending=False
    )
    .head(15)
)


# ------------------------------------------------
# Remove duplicate observations
# ------------------------------------------------

before_duplicates = len(df)

df = (
    df
    .drop_duplicates()
    .reset_index(drop=True)
)

after_duplicates = len(df)


print(
    "\nDuplicate rows removed:",
    before_duplicates
    - after_duplicates
)


# ------------------------------------------------
# Replace infinity values
# ------------------------------------------------

df = df.replace(
    [
        np.inf,
        -np.inf
    ],
    np.nan
)


# ================================================================
# 6. REMOVE NON-PREDICTIVE / LEAKAGE COLUMNS
# ================================================================

# Flow_ID is only an identifier.
#
# Label is the target.
#
# Binary_Label and Attack_Group contain information
# derived directly from the target.
#
# Synthetic_Anomaly_Score is also excluded if it exists
# because it should not be used to predict the class.

NON_FEATURE_COLUMNS = [

    "Flow_ID",

    "Binary_Label",

    "Attack_Group",

    "Synthetic_Anomaly_Score",

    "Label"

]


feature_columns = [

    column

    for column in df.columns

    if column
    not in NON_FEATURE_COLUMNS

]


print(
    "\nNumber of potential predictor variables:",
    len(feature_columns)
)


# ================================================================
# 7. CONVERT FEATURES TO NUMERIC
# ================================================================

for column in feature_columns:

    df[column] = pd.to_numeric(

        df[column],

        errors="coerce"

    )


# ================================================================
# 8. MISSING VALUE IMPUTATION
# ================================================================

for column in feature_columns:

    if df[column].isnull().sum() > 0:

        median_value = (
            df[column]
            .median()
        )

        df[column] = (
            df[column]
            .fillna(
                median_value
            )
        )


# ================================================================
# 9. REMOVE INVALID NEGATIVE VALUES
# ================================================================

for column in feature_columns:

    df.loc[
        df[column] < 0,
        column
    ] = 0


# ================================================================
# 10. DATA QUALITY SUMMARY
# ================================================================

quality_summary = pd.DataFrame({

    "Measure": [

        "Number of observations",

        "Number of variables",

        "Remaining missing values",

        "Remaining duplicate observations",

        "Number of traffic classes"

    ],

    "Value": [

        len(df),

        len(df.columns),

        df.isnull()
        .sum()
        .sum(),

        df.duplicated()
        .sum(),

        df["Label"]
        .nunique()

    ]

})


quality_summary.to_csv(

    os.path.join(

        TABLE_FOLDER,

        "data_quality_summary.csv"

    ),

    index=False

)


print(
    "\nData-quality summary:"
)

print(
    quality_summary
)


# ================================================================
# 11. EXPLORATORY DATA ANALYSIS
# ================================================================

print("\n================================================")
print("EXPLORATORY DATA ANALYSIS")
print("================================================")


# ================================================================
# FIGURE 1
# NETWORK TRAFFIC CLASS DISTRIBUTION
# ================================================================

class_counts = (
    df["Label"]
    .value_counts()
    .sort_values()
)


plt.figure(
    figsize=(11, 8)
)


class_counts.plot(
    kind="barh"
)


plt.title(
    "Distribution of Network Traffic Classes",
    fontsize=15
)


plt.xlabel(
    "Number of Network Flows"
)


plt.ylabel(
    "Traffic Class"
)


plt.tight_layout()


plt.savefig(

    os.path.join(

        FIGURE_FOLDER,

        "Figure_1_Class_Distribution.png"

    ),

    dpi=300

)


plt.close()


# ================================================================
# FIGURE 2
# BENIGN VS ATTACK
# ================================================================

binary_traffic = np.where(

    df["Label"]
    == "BENIGN",

    "BENIGN",

    "ATTACK"

)


binary_counts = (

    pd.Series(binary_traffic)

    .value_counts()

)


plt.figure(
    figsize=(7, 5)
)


binary_counts.plot(
    kind="bar"
)


plt.title(
    "Benign versus Malicious Network Traffic"
)


plt.xlabel(
    "Traffic Category"
)


plt.ylabel(
    "Number of Network Flows"
)


plt.xticks(
    rotation=0
)


plt.tight_layout()


plt.savefig(

    os.path.join(

        FIGURE_FOLDER,

        "Figure_2_Benign_vs_Attack.png"

    ),

    dpi=300

)


plt.close()


# ================================================================
# FIGURE 3
# PACKET RATE BY TRAFFIC CLASS
# ================================================================

major_classes = (

    df["Label"]
    .value_counts()
    .head(8)
    .index

)


major_df = df[

    df["Label"]
    .isin(
        major_classes
    )

].copy()


major_df[
    "Log_Flow_Packets_s"
] = np.log1p(

    major_df[
        "Flow_Packets_s"
    ]

)


packet_rate = (

    major_df

    .groupby(
        "Label"
    )[
        "Log_Flow_Packets_s"
    ]

    .median()

    .sort_values()

)


plt.figure(
    figsize=(11, 7)
)


packet_rate.plot(
    kind="barh"
)


plt.title(
    "Median Packet Rate by Major Network Traffic Class"
)


plt.xlabel(
    "log(1 + Flow Packets per Second)"
)


plt.ylabel(
    "Traffic Class"
)


plt.tight_layout()


plt.savefig(

    os.path.join(

        FIGURE_FOLDER,

        "Figure_3_Packet_Rate.png"

    ),

    dpi=300

)


plt.close()


# ================================================================
# FIGURE 4
# FLOW DURATION
# ================================================================

major_df[
    "Log_Flow_Duration"
] = np.log1p(

    major_df[
        "Flow_Duration_us"
    ]

)


duration_result = (

    major_df

    .groupby(
        "Label"
    )[
        "Log_Flow_Duration"
    ]

    .median()

    .sort_values()

)


plt.figure(
    figsize=(11, 7)
)


duration_result.plot(
    kind="barh"
)


plt.title(
    "Median Flow Duration by Major Traffic Class"
)


plt.xlabel(
    "log(1 + Flow Duration)"
)


plt.ylabel(
    "Traffic Class"
)


plt.tight_layout()


plt.savefig(

    os.path.join(

        FIGURE_FOLDER,

        "Figure_4_Flow_Duration.png"

    ),

    dpi=300

)


plt.close()


# ================================================================
# FIGURE 5
# TCP FLAG ANALYSIS
# ================================================================

flag_columns = [

    "SYN_Flag_Count",

    "RST_Flag_Count",

    "PSH_Flag_Count",

    "ACK_Flag_Count",

    "FIN_Flag_Count"

]


existing_flag_columns = [

    column

    for column
    in flag_columns

    if column
    in df.columns

]


flag_profile = (

    df

    .groupby(
        "Label"
    )[
        existing_flag_columns
    ]

    .mean()

)


flag_profile = (
    flag_profile
    .loc[
        major_classes
    ]
)


plt.figure(
    figsize=(12, 8)
)


image = plt.imshow(

    flag_profile.values,

    aspect="auto"

)


plt.colorbar(
    image,
    label="Mean Flag Count"
)


plt.xticks(

    range(
        len(
            existing_flag_columns
        )
    ),

    existing_flag_columns,

    rotation=35,

    ha="right"

)


plt.yticks(

    range(
        len(
            flag_profile.index
        )
    ),

    flag_profile.index

)


plt.title(
    "TCP Flag Behaviour across Network Traffic Classes"
)


plt.tight_layout()


plt.savefig(

    os.path.join(

        FIGURE_FOLDER,

        "Figure_5_TCP_Flag_Heatmap.png"

    ),

    dpi=300

)


plt.close()


# ================================================================
# FIGURE 6
# DESTINATION PORT ANALYSIS
# ================================================================

top_ports = (

    df[
        "Destination_Port"
    ]

    .value_counts()

    .head(15)

    .sort_values()

)


plt.figure(
    figsize=(10, 7)
)


top_ports.plot(
    kind="barh"
)


plt.title(
    "Top 15 Destination Ports"
)


plt.xlabel(
    "Number of Network Flows"
)


plt.ylabel(
    "Destination Port"
)


plt.tight_layout()


plt.savefig(

    os.path.join(

        FIGURE_FOLDER,

        "Figure_6_Destination_Ports.png"

    ),

    dpi=300

)


plt.close()


# ================================================================
# FIGURE 7
# CORRELATION MATRIX
# ================================================================

correlation_columns = [

    "Flow_Duration_us",

    "Total_Fwd_Packets",

    "Total_Backward_Packets",

    "Total_Length_Fwd_Packets",

    "Total_Length_Bwd_Packets",

    "Flow_Bytes_s",

    "Flow_Packets_s",

    "Flow_IAT_Mean_us",

    "Packet_Length_Mean",

    "Packet_Length_Std",

    "SYN_Flag_Count",

    "RST_Flag_Count",

    "ACK_Flag_Count"

]


correlation_columns = [

    column

    for column
    in correlation_columns

    if column
    in df.columns

]


correlation_matrix = (

    df[
        correlation_columns
    ]

    .corr()

)


plt.figure(
    figsize=(13, 10)
)


correlation_image = plt.imshow(

    correlation_matrix,

    vmin=-1,

    vmax=1,

    aspect="auto"

)


plt.colorbar(

    correlation_image,

    label="Pearson Correlation"

)


plt.xticks(

    range(
        len(
            correlation_columns
        )
    ),

    correlation_columns,

    rotation=75,

    ha="right",

    fontsize=8

)


plt.yticks(

    range(
        len(
            correlation_columns
        )
    ),

    correlation_columns,

    fontsize=8

)


plt.title(
    "Correlation Matrix of Network-Flow Features"
)


plt.tight_layout()


plt.savefig(

    os.path.join(

        FIGURE_FOLDER,

        "Figure_7_Correlation_Matrix.png"

    ),

    dpi=300

)


plt.close()


# ================================================================
# 12. CREATE X AND Y
# ================================================================

print("\n================================================")
print("STEP 3: FEATURE SELECTION AND NORMALISATION")
print("================================================")


X = df[
    feature_columns
].copy()


y_text = (

    df["Label"]
    .astype(str)

)


# ================================================================
# 13. ENCODE TARGET LABEL
# ================================================================

label_encoder = LabelEncoder()


y = label_encoder.fit_transform(
    y_text
)


print(
    "\nEncoded classes:"
)


for number, name in enumerate(
    label_encoder.classes_
):

    print(
        number,
        "=",
        name
    )


# ================================================================
# 14. TRAIN / TEST SPLIT
# ================================================================

X_train, X_test, y_train, y_test = (

    train_test_split(

        X,

        y,

        test_size=TEST_SIZE,

        random_state=RANDOM_STATE,

        stratify=y

    )

)


print(
    "\nTraining observations:",
    X_train.shape[0]
)


print(
    "Testing observations:",
    X_test.shape[0]
)


# ================================================================
# 15. FEATURE SELECTION
# ================================================================

number_features = min(

    TOP_K_FEATURES,

    X_train.shape[1]

)


feature_selector = SelectKBest(

    score_func=f_classif,

    k=number_features

)


X_train_selected = (

    feature_selector

    .fit_transform(

        X_train,

        y_train

    )

)


X_test_selected = (

    feature_selector

    .transform(
        X_test
    )

)


selected_features = (

    X_train.columns[

        feature_selector
        .get_support()

    ]

    .tolist()

)


print(
    "\nSelected features:"
)


for feature in selected_features:

    print(
        feature
    )


# ================================================================
# SAVE FEATURE SCORES
# ================================================================

feature_score_table = pd.DataFrame({

    "Feature":
        selected_features,

    "ANOVA_F_Score":
        feature_selector
        .scores_[

            feature_selector
            .get_support()

        ]

})


feature_score_table = (

    feature_score_table

    .sort_values(

        "ANOVA_F_Score",

        ascending=False

    )

)


feature_score_table.to_csv(

    os.path.join(

        TABLE_FOLDER,

        "selected_features.csv"

    ),

    index=False

)


# ================================================================
# 16. NORMALISATION
# ================================================================

scaler = StandardScaler()


X_train_scaled = (

    scaler

    .fit_transform(

        X_train_selected

    )

)


X_test_scaled = (

    scaler

    .transform(

        X_test_selected

    )

)


# Save preprocessing objects

joblib.dump(

    feature_selector,

    os.path.join(

        MODEL_FOLDER,

        "feature_selector.pkl"

    )

)


joblib.dump(

    scaler,

    os.path.join(

        MODEL_FOLDER,

        "scaler.pkl"

    )

)


joblib.dump(

    label_encoder,

    os.path.join(

        MODEL_FOLDER,

        "label_encoder.pkl"

    )

)


# ================================================================
# 17. DEFINE RANDOM FOREST
# ================================================================

print("\n================================================")
print("STEP 4: MACHINE LEARNING MODEL DEVELOPMENT")
print("================================================")


random_forest = RandomForestClassifier(

    n_estimators=300,

    class_weight="balanced",

    random_state=RANDOM_STATE,

    n_jobs=-1

)


# ================================================================
# 18. DEFINE XGBOOST
# ================================================================

xgboost_model = XGBClassifier(

    n_estimators=300,

    max_depth=6,

    learning_rate=0.08,

    subsample=0.85,

    colsample_bytree=0.85,

    objective="multi:softprob",

    num_class=len(
        label_encoder.classes_
    ),

    eval_metric="mlogloss",

    random_state=RANDOM_STATE,

    n_jobs=-1

)


# ================================================================
# 19. DEFINE SVM
# ================================================================

svm_model = SVC(

    kernel="rbf",

    C=5,

    gamma="scale",

    class_weight="balanced",

    probability=True,

    random_state=RANDOM_STATE

)


# ================================================================
# 20. MODEL EVALUATION FUNCTION
# ================================================================

def evaluate_model(

    model_name,

    model,

    X_train_model,

    X_test_model

):


    print(
        "\n================================================"
    )

    print(
        "TRAINING:",
        model_name
    )

    print(
        "================================================"
    )


    # Train model

    model.fit(

        X_train_model,

        y_train

    )


    # Prediction

    y_prediction = model.predict(

        X_test_model

    )


    # Probability prediction

    prediction_probability = (

        model.predict_proba(

            X_test_model

        )

    )


    # ------------------------------------------------
    # Accuracy
    # ------------------------------------------------

    accuracy = accuracy_score(

        y_test,

        y_prediction

    )


    # ------------------------------------------------
    # Precision
    # ------------------------------------------------

    precision_macro = precision_score(

        y_test,

        y_prediction,

        average="macro",

        zero_division=0

    )


    precision_weighted = precision_score(

        y_test,

        y_prediction,

        average="weighted",

        zero_division=0

    )


    # ------------------------------------------------
    # Recall
    # ------------------------------------------------

    recall_macro = recall_score(

        y_test,

        y_prediction,

        average="macro",

        zero_division=0

    )


    recall_weighted = recall_score(

        y_test,

        y_prediction,

        average="weighted",

        zero_division=0

    )


    # ------------------------------------------------
    # F1
    # ------------------------------------------------

    f1_macro = f1_score(

        y_test,

        y_prediction,

        average="macro",

        zero_division=0

    )


    f1_weighted = f1_score(

        y_test,

        y_prediction,

        average="weighted",

        zero_division=0

    )


    # ------------------------------------------------
    # ROC-AUC
    # ------------------------------------------------

    try:

        roc_auc = roc_auc_score(

            y_test,

            prediction_probability,

            multi_class="ovr",

            average="macro"

        )

    except:

        roc_auc = np.nan


    print(
        "\nAccuracy:",
        round(
            accuracy,
            4
        )
    )


    print(
        "Macro Precision:",
        round(
            precision_macro,
            4
        )
    )


    print(
        "Macro Recall:",
        round(
            recall_macro,
            4
        )
    )


    print(
        "Macro F1:",
        round(
            f1_macro,
            4
        )
    )


    print(
        "ROC-AUC:",
        round(
            roc_auc,
            4
        )
    )


    # ================================================================
    # CLASSIFICATION REPORT
    # ================================================================

    report = classification_report(

        y_test,

        y_prediction,

        target_names=
        label_encoder.classes_,

        output_dict=True,

        zero_division=0

    )


    report_dataframe = (

        pd.DataFrame(
            report
        )

        .transpose()

    )


    safe_name = (

        model_name

        .replace(
            " ",
            "_"
        )

        .lower()

    )


    report_dataframe.to_csv(

        os.path.join(

            TABLE_FOLDER,

            safe_name
            + "_classification_report.csv"

        )

    )


    # ================================================================
    # CONFUSION MATRIX
    # ================================================================

    matrix = confusion_matrix(

        y_test,

        y_prediction

    )


    plt.figure(
        figsize=(13, 10)
    )


    matrix_image = plt.imshow(

        matrix,

        aspect="auto"

    )


    plt.colorbar(
        matrix_image
    )


    tick_locations = np.arange(

        len(
            label_encoder.classes_
        )

    )


    plt.xticks(

        tick_locations,

        label_encoder.classes_,

        rotation=75,

        ha="right",

        fontsize=7

    )


    plt.yticks(

        tick_locations,

        label_encoder.classes_,

        fontsize=7

    )


    plt.xlabel(
        "Predicted Traffic Class"
    )


    plt.ylabel(
        "Actual Traffic Class"
    )


    plt.title(

        model_name
        + " Confusion Matrix"

    )


    plt.tight_layout()


    plt.savefig(

        os.path.join(

            FIGURE_FOLDER,

            safe_name
            + "_confusion_matrix.png"

        ),

        dpi=300

    )


    plt.close()


    # ================================================================
    # F1 SCORE BY CLASS
    # ================================================================

    class_report = (

        report_dataframe

        .loc[
            label_encoder.classes_
        ]

    )


    f1_class = (

        class_report[
            "f1-score"
        ]

        .sort_values()

    )


    plt.figure(
        figsize=(11, 8)
    )


    f1_class.plot(
        kind="barh"
    )


    plt.title(

        model_name
        + " F1 Score by Traffic Class"

    )


    plt.xlabel(
        "F1 Score"
    )


    plt.ylabel(
        "Traffic Class"
    )


    plt.xlim(
        0,
        1
    )


    plt.tight_layout()


    plt.savefig(

        os.path.join(

            FIGURE_FOLDER,

            safe_name
            + "_f1_by_class.png"

        ),

        dpi=300

    )


    plt.close()


    # ================================================================
    # MULTICLASS ROC CURVES
    # ================================================================

    y_test_binary = label_binarize(

        y_test,

        classes=np.arange(

            len(
                label_encoder.classes_
            )

        )

    )


    plt.figure(
        figsize=(10, 8)
    )


    for class_number, class_name in enumerate(

        label_encoder.classes_

    ):


        false_positive_rate, true_positive_rate, _ = (

            roc_curve(

                y_test_binary[
                    :,
                    class_number
                ],

                prediction_probability[
                    :,
                    class_number
                ]

            )

        )


        class_auc = auc(

            false_positive_rate,

            true_positive_rate

        )


        plt.plot(

            false_positive_rate,

            true_positive_rate,

            label=(

                class_name

                + " AUC="

                + str(
                    round(
                        class_auc,
                        3
                    )
                )

            )

        )


    plt.plot(

        [0, 1],

        [0, 1],

        linestyle="--"

    )


    plt.xlabel(
        "False Positive Rate"
    )


    plt.ylabel(
        "True Positive Rate"
    )


    plt.title(

        model_name
        + " Multiclass ROC Curves"

    )


    plt.legend(

        fontsize=6,

        loc="lower right"

    )


    plt.tight_layout()


    plt.savefig(

        os.path.join(

            FIGURE_FOLDER,

            safe_name
            + "_ROC_curves.png"

        ),

        dpi=300

    )


    plt.close()


    # ================================================================
    # PRECISION-RECALL CURVES
    # ================================================================

    plt.figure(
        figsize=(10, 8)
    )


    for class_number, class_name in enumerate(

        label_encoder.classes_

    ):


        precision_values, recall_values, _ = (

            precision_recall_curve(

                y_test_binary[
                    :,
                    class_number
                ],

                prediction_probability[
                    :,
                    class_number
                ]

            )

        )


        AP = average_precision_score(

            y_test_binary[
                :,
                class_number
            ],

            prediction_probability[
                :,
                class_number
            ]

        )


        plt.plot(

            recall_values,

            precision_values,

            label=(

                class_name

                + " AP="

                + str(
                    round(
                        AP,
                        3
                    )
                )

            )

        )


    plt.xlabel(
        "Recall"
    )


    plt.ylabel(
        "Precision"
    )


    plt.title(

        model_name
        + " Precision-Recall Curves"

    )


    plt.legend(

        fontsize=6,

        loc="lower left"

    )


    plt.tight_layout()


    plt.savefig(

        os.path.join(

            FIGURE_FOLDER,

            safe_name
            + "_Precision_Recall.png"

        ),

        dpi=300

    )


    plt.close()


    # ================================================================
    # SAVE MODEL
    # ================================================================

    joblib.dump(

        model,

        os.path.join(

            MODEL_FOLDER,

            safe_name
            + ".pkl"

        )

    )


    result = {

        "Model":
            model_name,

        "Accuracy":
            accuracy,

        "Precision_Macro":
            precision_macro,

        "Recall_Macro":
            recall_macro,

        "F1_Macro":
            f1_macro,

        "Precision_Weighted":
            precision_weighted,

        "Recall_Weighted":
            recall_weighted,

        "F1_Weighted":
            f1_weighted,

        "ROC_AUC":
            roc_auc

    }


    return (

        result,

        y_prediction,

        prediction_probability

    )


# ================================================================
# 21. TRAIN RANDOM FOREST
# ================================================================

RF_result, RF_prediction, RF_probability = (

    evaluate_model(

        "Random Forest",

        random_forest,

        X_train_selected,

        X_test_selected

    )

)


# ================================================================
# 22. TRAIN XGBOOST
# ================================================================

XGB_result, XGB_prediction, XGB_probability = (

    evaluate_model(

        "XGBoost",

        xgboost_model,

        X_train_selected,

        X_test_selected

    )

)


# ================================================================
# 23. TRAIN SVM
# ================================================================

SVM_result, SVM_prediction, SVM_probability = (

    evaluate_model(

        "SVM",

        svm_model,

        X_train_scaled,

        X_test_scaled

    )

)


# ================================================================
# 24. MODEL PERFORMANCE COMPARISON
# ================================================================

results_dataframe = pd.DataFrame([

    RF_result,

    XGB_result,

    SVM_result

])


results_dataframe.to_csv(

    os.path.join(

        TABLE_FOLDER,

        "model_performance_comparison.csv"

    ),

    index=False

)


print("\n================================================")
print("FINAL MODEL COMPARISON")
print("================================================")


print(
    results_dataframe
    .round(4)
)


# ================================================================
# FIGURE 8
# MODEL PERFORMANCE COMPARISON
# ================================================================

comparison_metrics = [

    "Accuracy",

    "Precision_Macro",

    "Recall_Macro",

    "F1_Macro"

]


comparison_plot = (

    results_dataframe

    .set_index(
        "Model"
    )[
        comparison_metrics
    ]

)


comparison_plot.plot(

    kind="bar",

    figsize=(11, 7)

)


plt.ylim(
    0,
    1
)


plt.ylabel(
    "Performance Score"
)


plt.xlabel(
    "Machine Learning Algorithm"
)


plt.title(
    "Comparison of Intrusion Detection Model Performance"
)


plt.xticks(
    rotation=0
)


plt.tight_layout()


plt.savefig(

    os.path.join(

        FIGURE_FOLDER,

        "Figure_8_Model_Performance_Comparison.png"

    ),

    dpi=300

)


plt.close()


# ================================================================
# 25. RANDOM FOREST FEATURE IMPORTANCE
# ================================================================

print("\n================================================")
print("RANDOM FOREST FEATURE IMPORTANCE")
print("================================================")


RF_feature_importance = pd.Series(

    random_forest.feature_importances_,

    index=selected_features

)


RF_feature_importance = (

    RF_feature_importance

    .sort_values(
        ascending=False
    )

)


print(
    RF_feature_importance
)


RF_feature_importance.to_csv(

    os.path.join(

        TABLE_FOLDER,

        "Random_Forest_Feature_Importance.csv"

    ),

    header=[
        "Importance"
    ]

)


# ================================================================
# FIGURE 9
# RANDOM FOREST FEATURE IMPORTANCE
# ================================================================

plt.figure(
    figsize=(11, 8)
)


RF_feature_importance.sort_values().plot(

    kind="barh"

)


plt.title(
    "Random Forest Feature Importance"
)


plt.xlabel(
    "Feature Importance"
)


plt.ylabel(
    "Network Traffic Feature"
)


plt.tight_layout()


plt.savefig(

    os.path.join(

        FIGURE_FOLDER,

        "Figure_9_RF_Feature_Importance.png"

    ),

    dpi=300

)


plt.close()


# ================================================================
# 26. SHAP EXPLAINABILITY
# ================================================================

print("\n================================================")
print("STEP 5: SHAP EXPLAINABILITY")
print("================================================")


if SHAP_AVAILABLE:


    # Use a sample to reduce computation time

    SHAP_SAMPLE_SIZE = min(

        500,

        len(
            X_test_selected
        )

    )


    X_SHAP = pd.DataFrame(

        X_test_selected[
            :SHAP_SAMPLE_SIZE
        ],

        columns=selected_features

    )


    try:


        shap_explainer = (

            shap.TreeExplainer(

                random_forest

            )

        )


        shap_values = (

            shap_explainer
            .shap_values(

                X_SHAP

            )

        )


        # ------------------------------------------------
        # SHAP SUMMARY
        # ------------------------------------------------

        shap.summary_plot(

            shap_values,

            X_SHAP,

            feature_names=
            selected_features,

            show=False,

            max_display=15

        )


        plt.title(
            "SHAP Summary Plot – Random Forest"
        )


        plt.tight_layout()


        plt.savefig(

            os.path.join(

                FIGURE_FOLDER,

                "Figure_10_SHAP_Summary.png"

            ),

            dpi=300,

            bbox_inches="tight"

        )


        plt.close()


        print(
            "SHAP explanation generated successfully."
        )


    except Exception as error:


        print(
            "Standard SHAP summary error:",
            error
        )


        print(
            "Generating SHAP global importance instead."
        )


        shap_values_array = (

            np.asarray(
                shap_values
            )

        )


        # Multiclass handling

        if isinstance(
            shap_values,
            list
        ):


            mean_shap = np.mean(

                np.stack([

                    np.abs(
                        values
                    ).mean(axis=0)

                    for values
                    in shap_values

                ]),

                axis=0

            )


        elif (

            shap_values_array.ndim
            == 3

        ):


            if (

                shap_values_array.shape[1]
                ==
                len(
                    selected_features
                )

            ):


                mean_shap = (

                    np.abs(
                        shap_values_array
                    )

                    .mean(
                        axis=(0, 2)
                    )

                )


            else:


                mean_shap = (

                    np.abs(
                        shap_values_array
                    )

                    .mean(
                        axis=(0, 1)
                    )

                )


        else:


            mean_shap = (

                np.abs(
                    shap_values_array
                )

                .mean(
                    axis=0
                )

            )


        SHAP_importance = pd.Series(

            mean_shap,

            index=selected_features

        )


        SHAP_importance = (

            SHAP_importance

            .sort_values(
                ascending=False
            )

        )


        SHAP_importance.to_csv(

            os.path.join(

                TABLE_FOLDER,

                "SHAP_Global_Importance.csv"

            ),

            header=[
                "Mean_Absolute_SHAP"
            ]

        )


        plt.figure(
            figsize=(11, 8)
        )


        SHAP_importance.head(15).sort_values().plot(

            kind="barh"

        )


        plt.title(
            "SHAP Global Feature Importance"
        )


        plt.xlabel(
            "Mean Absolute SHAP Value"
        )


        plt.ylabel(
            "Network Traffic Feature"
        )


        plt.tight_layout()


        plt.savefig(

            os.path.join(

                FIGURE_FOLDER,

                "Figure_10_SHAP_Global_Importance.png"

            ),

            dpi=300

        )


        plt.close()


else:


    print(
        "SHAP analysis skipped."
    )


# ================================================================
# 27. LIME LOCAL EXPLANATION
# ================================================================

print("\n================================================")
print("STEP 6: LIME EXPLAINABILITY")
print("================================================")


if LIME_AVAILABLE:


    try:


        lime_explainer = LimeTabularExplainer(

            training_data=
            np.asarray(
                X_train_selected
            ),

            feature_names=
            selected_features,

            class_names=
            list(
                label_encoder.classes_
            ),

            mode=
            "classification",

            discretize_continuous=
            True,

            random_state=
            RANDOM_STATE

        )


        # Explain first observation in test data

        test_instance = (

            X_test_selected[
                0
            ]

        )


        lime_explanation = (

            lime_explainer

            .explain_instance(

                test_instance,

                random_forest
                .predict_proba,

                num_features=min(

                    12,

                    len(
                        selected_features
                    )

                )

            )

        )


        lime_results = pd.DataFrame(

            lime_explanation
            .as_list(),

            columns=[

                "Feature_Condition",

                "Contribution"

            ]

        )


        lime_results.to_csv(

            os.path.join(

                TABLE_FOLDER,

                "LIME_Local_Explanation.csv"

            ),

            index=False

        )


        print(
            "\nLIME explanation:"
        )


        print(
            lime_results
        )


        # ================================================================
        # FIGURE 11
        # LIME LOCAL EXPLANATION
        # ================================================================

        plot_lime = (

            lime_results

            .sort_values(
                "Contribution"
            )

        )


        plt.figure(
            figsize=(11, 8)
        )


        plt.barh(

            plot_lime[
                "Feature_Condition"
            ],

            plot_lime[
                "Contribution"
            ]

        )


        plt.axvline(
            0,
            linewidth=1
        )


        plt.title(
            "LIME Local Explanation for an Intrusion Prediction"
        )


        plt.xlabel(
            "Contribution to Model Prediction"
        )


        plt.ylabel(
            "Feature Condition"
        )


        plt.tight_layout()


        plt.savefig(

            os.path.join(

                FIGURE_FOLDER,

                "Figure_11_LIME_Local_Explanation.png"

            ),

            dpi=300

        )


        plt.close()


        print(
            "LIME explanation successfully generated."
        )


    except Exception as error:


        print(
            "LIME error:",
            error
        )


else:


    print(
        "LIME analysis skipped."
    )


# ================================================================
# 28. INTRUSION PREDICTION RESULTS
# ================================================================

print("\n================================================")
print("STEP 7: INTRUSION PREDICTION")
print("================================================")


prediction_table = (

    df

    .loc[
        X_test.index
    ]

    .copy()

)


prediction_table[
    "Actual_Class"
] = label_encoder.inverse_transform(

    y_test

)


prediction_table[
    "RF_Prediction"
] = label_encoder.inverse_transform(

    RF_prediction

)


prediction_table[
    "XGBoost_Prediction"
] = label_encoder.inverse_transform(

    XGB_prediction

)


prediction_table[
    "SVM_Prediction"
] = label_encoder.inverse_transform(

    SVM_prediction

)


prediction_table[
    "RF_Confidence"
] = (

    RF_probability

    .max(
        axis=1
    )

)


prediction_table[
    "XGBoost_Confidence"
] = (

    XGB_probability

    .max(
        axis=1
    )

)


prediction_table[
    "SVM_Confidence"
] = (

    SVM_probability

    .max(
        axis=1
    )

)


# ================================================================
# 29. DECISION SUPPORT ALERT LEVEL
# ================================================================

def create_alert_level(

    predicted_class,

    confidence

):


    if predicted_class == "BENIGN":

        return "LOW"


    elif confidence >= 0.90:

        return "CRITICAL"


    elif confidence >= 0.75:

        return "HIGH"


    elif confidence >= 0.60:

        return "MEDIUM"


    else:

        return "REVIEW"


prediction_table[
    "Analyst_Alert_Level"
] = [

    create_alert_level(

        prediction,

        confidence

    )

    for prediction, confidence
    in zip(

        prediction_table[
            "XGBoost_Prediction"
        ],

        prediction_table[
            "XGBoost_Confidence"
        ]

    )

]


# ================================================================
# 30. SAVE ANALYST DASHBOARD DATA
# ================================================================

dashboard_columns = [

    "Flow_ID",

    "Destination_Port",

    "Protocol",

    "Flow_Duration_us",

    "Flow_Bytes_s",

    "Flow_Packets_s",

    "Actual_Class",

    "RF_Prediction",

    "XGBoost_Prediction",

    "SVM_Prediction",

    "RF_Confidence",

    "XGBoost_Confidence",

    "SVM_Confidence",

    "Analyst_Alert_Level"

]


dashboard_columns = [

    column

    for column
    in dashboard_columns

    if column
    in prediction_table.columns

]


dashboard_data = (

    prediction_table[
        dashboard_columns
    ]

)


dashboard_data.to_csv(

    os.path.join(

        OUTPUT_FOLDER,

        "analyst_dashboard_predictions.csv"

    ),

    index=False

)


print(
    "\nAnalyst dashboard data successfully exported."
)


# ================================================================
# 31. HIGH-RISK INTRUSION ALERTS
# ================================================================

high_risk_alerts = (

    dashboard_data[

        dashboard_data[
            "Analyst_Alert_Level"
        ]

        .isin([

            "CRITICAL",

            "HIGH"

        ])

    ]

)


high_risk_alerts.to_csv(

    os.path.join(

        OUTPUT_FOLDER,

        "high_risk_intrusion_alerts.csv"

    ),

    index=False

)


print(
    "\nNumber of high/critical alerts:",
    len(
        high_risk_alerts
    )
)


# ================================================================
# 32. MODEL RANKING
# ================================================================

model_ranking = (

    results_dataframe

    .sort_values(

        "F1_Macro",

        ascending=False

    )

)


model_ranking.to_csv(

    os.path.join(

        TABLE_FOLDER,

        "model_ranking.csv"

    ),

    index=False

)


print("\n================================================")
print("MODEL RANKING")
print("================================================")


print(

    model_ranking[[

        "Model",

        "Accuracy",

        "Precision_Macro",

        "Recall_Macro",

        "F1_Macro",

        "ROC_AUC"

    ]]

    .round(4)

)


# ================================================================
# 33. IDENTIFY BEST MODEL
# ================================================================

best_model = (

    model_ranking

    .iloc[0][
        "Model"
    ]

)


best_f1 = (

    model_ranking

    .iloc[0][
        "F1_Macro"
    ]

)


print(
    "\nBest overall model:",
    best_model
)


print(
    "Best Macro F1:",
    round(
        best_f1,
        4
    )
)


# ================================================================
# 34. SAVE PROJECT SUMMARY
# ================================================================

project_summary = {

    "Dataset":
        DATA_FILE,

    "Number_of_Records":
        int(
            len(df)
        ),

    "Number_of_Traffic_Classes":
        int(
            df["Label"]
            .nunique()
        ),

    "Training_Size":
        int(
            len(
                X_train
            )
        ),

    "Testing_Size":
        int(
            len(
                X_test
            )
        ),

    "Selected_Features":
        selected_features,

    "Machine_Learning_Models": [

        "Random Forest",

        "XGBoost",

        "Support Vector Machine"

    ],

    "Explainability_Methods": [

        "SHAP",

        "LIME"

    ],

    "Best_Model":
        best_model,

    "Best_Macro_F1":
        float(
            best_f1
        )

}


with open(

    os.path.join(

        OUTPUT_FOLDER,

        "project_summary.json"

    ),

    "w"

) as file:


    json.dump(

        project_summary,

        file,

        indent=4

    )


# ================================================================
# 35. FINAL OUTPUT
# ================================================================

print("\n================================================")
print("PROJECT EXECUTION COMPLETED SUCCESSFULLY")
print("================================================")


print(
    "\nThe following stages have been completed:"
)


print(
    """
    1. CICIDS2017 dataset loading
    2. Data cleaning
    3. Missing-value treatment
    4. Duplicate removal
    5. Feature preparation
    6. ANOVA-based feature selection
    7. Data normalisation
    8. Random Forest modelling
    9. XGBoost modelling
    10. SVM modelling
    11. Intrusion classification
    12. Accuracy evaluation
    13. Precision evaluation
    14. Recall evaluation
    15. F1-score evaluation
    16. ROC-AUC analysis
    17. Confusion matrices
    18. ROC curves
    19. Precision-Recall curves
    20. Random Forest feature importance
    21. SHAP explainability
    22. LIME explainability
    23. Intrusion prediction confidence
    24. Analyst alert levels
    25. Decision-support dataset
    """
)


print(
    "\nAll graphs are stored in:",
    FIGURE_FOLDER
)


print(
    "All analytical tables are stored in:",
    TABLE_FOLDER
)


print(
    "All trained models are stored in:",
    MODEL_FOLDER
)


print(
    "\nProject completed."
)


STEP 1: DATASET COLLECTION

Dataset successfully loaded.

Dataset dimensions: (30000, 37)

Number of rows: 30000

Number of columns: 37

Dataset columns:

Flow_ID
Destination_Port
Protocol
Flow_Duration_us
Total_Fwd_Packets
Total_Backward_Packets
Total_Length_Fwd_Packets
Total_Length_Bwd_Packets
Fwd_Packet_Length_Mean
Bwd_Packet_Length_Mean
Flow_Bytes_s
Flow_Packets_s
Flow_IAT_Mean_us
Flow_IAT_Std_us
Fwd_IAT_Mean_us
Bwd_IAT_Mean_us
Fwd_Header_Length
Bwd_Header_Length
Fwd_Packets_s
Bwd_Packets_s
Min_Packet_Length
Max_Packet_Length
Packet_Length_Mean
Packet_Length_Std
FIN_Flag_Count
SYN_Flag_Count
RST_Flag_Count
PSH_Flag_Count
ACK_Flag_Count
URG_Flag_Count
Average_Packet_Size
Avg_Fwd_Segment_Size
Avg_Bwd_Segment_Size
Synthetic_Anomaly_Score
Binary_Label
Attack_Group
Label

First five observations:

     Flow_ID  Destination_Port  Protocol  Flow_Duration_us  Total_Fwd_Packets  \
0  SYN000001              8000         6       66626963.05                 17   
1  SYN000002             3117

In [9]:
!pip install google-genai

In [10]:
!pip install -q google-genai openai

In [27]:
!pip install -q google-genai anthropic openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 11.5 MB/s eta 0:00:00


In [34]:
# =============================================================================
# MULTI-PROVIDER MULTI-AGENT EXPLAINABILITY EVALUATION (Gemini + Claude + GPT)
# Runs the SAME 3 personas across 3 LLM providers = 9 agents
# Persona prompts and rubric are IDENTICAL across providers by design, so the
# only variable being measured is the underlying LLM.
#
# NOTE: This version uses LOCAL Colab storage only (no Google Drive mount).
# Local files are wiped when the runtime resets/disconnects, so download
# your results (outputs/results/multi_provider_agent_eval.csv / .txt) before
# ending the session, or re-upload your source CSVs each time you start a new one.
# =============================================================================

import os
import re
import pandas as pd
from google.colab import userdata

# -----------------------------------------------------------------------------
# 1. Load API keys from Colab Secrets
#    Make sure GEMINI_API_KEY, ANTHROPIC_API_KEY, and OPENAI_API_KEY are all
#    added in Colab Secrets (left sidebar, key icon) with the toggle ON.
#    NEVER paste API keys directly into a notebook cell or into chat — always
#    use Colab Secrets so the key stays out of your notebook file and version
#    control.
# -----------------------------------------------------------------------------
try:
    gemini_key = userdata.get('GEMINI_API_KEY')
    anthropic_key = userdata.get('ANTHROPIC_API_KEY')
    openai_key = userdata.get('OPENAI_API_KEY')
except Exception as e:
    raise ValueError(f"Could not load one or more API key secrets. Make sure all three toggles are ON in Colab Secrets! Error: {e}")

# Install SDKs if not already present (uncomment if needed in a fresh runtime)
# !pip install -q google-genai anthropic openai

from google import genai
import anthropic
import openai

gemini_client = genai.Client(api_key=gemini_key)
anthropic_client = anthropic.Anthropic(api_key=anthropic_key)
openai_client = openai.OpenAI(api_key=openai_key)

# -----------------------------------------------------------------------------
# 2. Model selection — verified as of Aug 2026. Update here if newer versions
#    are released before you run your final experiments; report the exact
#    model strings used in your Methodology chapter for reproducibility.
# -----------------------------------------------------------------------------
MODELS = {
    "Gemini": "gemini-3.5-flash",
    "Claude": "claude-sonnet-5",
    "GPT": "gpt-5.5",
}

print(f"Using Models: {MODELS}")

# -----------------------------------------------------------------------------
# 3. Local input/output folders (all within the Colab session, no Drive)
#    Upload your SHAP/LIME CSVs into INPUT_DIR before running this cell
#    (drag-and-drop into the Colab file browser on the left, into
#    outputs/tables/), or adjust the paths below to wherever you uploaded them.
# -----------------------------------------------------------------------------
INPUT_DIR = "outputs/tables"
OUTPUT_DIR = "outputs/results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

shap_path = os.path.join(INPUT_DIR, "Random_Forest_Feature_Importance.csv")
lime_path = os.path.join(INPUT_DIR, "LIME_Local_Explanation.csv")

if not os.path.exists(shap_path):
    raise FileNotFoundError(
        f"'{shap_path}' not found. Upload your SHAP CSV to this path in the Colab "
        f"file browser (left sidebar) before running this cell."
    )

shap_data = pd.read_csv(shap_path).head(10).to_string()

if os.path.exists(lime_path):
    lime_data = pd.read_csv(lime_path).to_string()
else:
    lime_data = "LIME output file not found. LIME analysis was likely skipped."
    print(f"Warning: {lime_path} not found. LIME data will be replaced with a placeholder.")

# -----------------------------------------------------------------------------
# 4. Define Expert Personas (IDENTICAL across all three providers)
# -----------------------------------------------------------------------------
personas = {
    "SOC_Manager": {
        "title": "Security Operations Center (SOC) Manager",
        "perspective": "You care about rapid threat triage, operational utility, and clear actionability."
    },
    "Compliance_Auditor": {
        "title": "IT Compliance & Risk Auditor",
        "perspective": "You care about auditability, consistency, transparency, and non-jargon explanations."
    },
    "CISO_Executive": {
        "title": "Chief Information Security Officer (Non-Technical Executive)",
        "perspective": "You care about high-level business risk, clarity, and plain-English summaries."
    }
}

# -----------------------------------------------------------------------------
# 5. Define Evaluation Prompt Template (IDENTICAL across all three providers)
# -----------------------------------------------------------------------------
prompt_template = """
You are acting as a {title}. {perspective}

Below are two Explainable AI (XAI) outputs explaining a machine learning intrusion detection alert:

--- SHAP OUTPUT (Global Feature Importances) ---
{shap_data}

--- LIME OUTPUT (Local Feature Conditions) ---
{lime_data}

EVALUATION TASK:
Do NOT evaluate model performance metrics (like F1 score or Accuracy). Focus strictly on EXPLAINABILITY.
1. Evaluate SHAP in terms of clarity, ease of understanding, and usefulness for your role.
2. Evaluate LIME in terms of clarity, ease of understanding, and usefulness for your role.
3. Declare which approach (SHAP or LIME) is MORE EXPLAINABLE and useful to you, providing 2 concrete reasons why.

Format your output strictly as:
- Persona: [{title}]
- SHAP Rating (1-10): [Score]
- LIME Rating (1-10): [Score]
- Preferred Method: [SHAP / LIME]
- Reason 1: [Brief explanation]
- Reason 2: [Brief explanation]
"""

# -----------------------------------------------------------------------------
# 6. Provider call wrappers — each returns plain text, same interface
# -----------------------------------------------------------------------------

def call_gemini(prompt):
    response = gemini_client.models.generate_content(
        model=MODELS["Gemini"],
        contents=prompt
    )
    return response.text

def call_claude(prompt):
    response = anthropic_client.messages.create(
        model=MODELS["Claude"],
        max_tokens=1000,
        messages=[{"role": "user", "content": prompt}]
    )
    # Claude can return multiple content blocks (e.g. a ThinkingBlock followed
    # by a TextBlock). Find the actual text block rather than assuming index 0.
    for block in response.content:
        if block.type == "text":
            return block.text
    raise ValueError("No text block found in Claude response.")

def call_gpt(prompt):
    response = openai_client.chat.completions.create(
        model=MODELS["GPT"],
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

providers = {
    "Gemini": call_gemini,
    "Claude": call_claude,
    "GPT": call_gpt,
}

# -----------------------------------------------------------------------------
# 7. Execute Multi-Provider Multi-Agent Evaluation (3 personas x 3 providers = 9)
# -----------------------------------------------------------------------------
evaluation_results = []   # raw text, for the .txt log
parsed_rows = []          # structured rows, for the .csv / analysis

# Fixed regex: anchors to the literal "(1-10)" so it captures the actual score,
# not the "1" inside "(1-10)" from the prompt template's own instructions.
rating_pattern = re.compile(
    r"SHAP Rating \(1-10\):\s*\[?(\d+)\]?.*?LIME Rating \(1-10\):\s*\[?(\d+)\]?.*?Preferred Method:\s*\[?(\w+)",
    re.DOTALL | re.IGNORECASE
)

print("==================================================")
print("RUNNING MULTI-PROVIDER MULTI-AGENT EXPLAINABILITY EVALUATION")
print("==================================================")

for provider_name, call_fn in providers.items():
    for persona_key, persona_info in personas.items():
        formatted_prompt = prompt_template.format(
            title=persona_info["title"],
            perspective=persona_info["perspective"],
            shap_data=shap_data,
            lime_data=lime_data
        )

        try:
            agent_output = call_fn(formatted_prompt)
        except Exception as e:
            agent_output = f"[ERROR generating response for {provider_name} / {persona_info['title']}: {e}]"

        header = f"=== Provider: {provider_name} | Persona: {persona_info['title']} ==="
        evaluation_results.append(f"{header}\n{agent_output}")

        print(f"\n--- {header} ---\n")
        print(agent_output)

        # Try to parse ratings for structured analysis; fall back to blanks on failure
        match = rating_pattern.search(agent_output)
        if match:
            shap_score, lime_score, preferred = match.groups()
        else:
            shap_score, lime_score, preferred = None, None, None

        parsed_rows.append({
            "Provider": provider_name,
            "Model": MODELS[provider_name],
            "Persona": persona_info["title"],
            "SHAP_Rating": shap_score,
            "LIME_Rating": lime_score,
            "Preferred_Method": preferred,
            "Raw_Output": agent_output,
        })

# -----------------------------------------------------------------------------
# 8. Save Results locally — raw text log + structured CSV
# -----------------------------------------------------------------------------
txt_path = os.path.join(OUTPUT_DIR, "multi_provider_agent_eval.txt")
csv_path = os.path.join(OUTPUT_DIR, "multi_provider_agent_eval.csv")

with open(txt_path, "w") as f:
    f.write("\n\n==========================================\n\n".join(evaluation_results))

results_df = pd.DataFrame(parsed_rows)
results_df.to_csv(csv_path, index=False)

print("\nEvaluation complete! Results saved locally to:")
print(f" - {txt_path} (raw responses)")
print(f" - {csv_path} (structured ratings for analysis)")
print("\nRemember: these are LOCAL Colab files — download them (right-click in the")
print("file browser > Download) before this runtime disconnects, or they'll be lost.")

# Quick sanity-check summary table
print("\nSummary of ratings collected:")
print(results_df[["Provider", "Persona", "SHAP_Rating", "LIME_Rating", "Preferred_Method"]])

Using Models: {'Gemini': 'gemini-3.5-flash', 'Claude': 'claude-sonnet-5', 'GPT': 'gpt-5.5'}
RUNNING MULTI-PROVIDER MULTI-AGENT EXPLAINABILITY EVALUATION

--- === Provider: Gemini | Persona: Security Operations Center (SOC) Manager === ---

- Persona: Security Operations Center (SOC) Manager
- SHAP Rating (1-10): 4
- LIME Rating (1-10): 8
- Preferred Method: LIME
- Reason 1: Localized Specificity for Incident Triage. LIME explains the exact conditions of the specific network flow that triggered this alert, whereas SHAP's global features only show what the model values generally, which does not help triage a single, active ticket.
- Reason 2: Actionable Defense and Rule Writing. LIME provides explicit numerical ranges (e.g., packet rates and packet lengths) that my analysts can immediately translate into firewall rules, Snort signatures, or PCAP filters for rapid containment and investigation.

--- === Provider: Gemini | Persona: IT Compliance & Risk Auditor === ---

Persona: [IT Complia